In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

url = "https://www.fcpeuro.com/products/porsche-tie-rod-kit-lemforder-3694601kt#fitment"

options = webdriver.ChromeOptions()
options.add_argument("--headless")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
driver.get(url)

wait = WebDriverWait(driver, 12)

# Wait for Fitment panel
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.tabs-panel.is-active")))
time.sleep(2)

# Parse page
html = driver.page_source
soup = BeautifulSoup(html, "html.parser")

active_panel = soup.select_one("div.tabs-panel.is-active")

if active_panel:

    panel_text = active_panel.get_text("\n", strip=True)
    lines = [line.strip() for line in panel_text.split("\n") if line.strip()]

    data = []

    for line in lines:
        # Look for a year at the beginning of the line (e.g., "2007 BMW 335i Base Sedan")
        match = re.match(r"(^20\d{2})", line)
        year = match.group(1) if match else ""

        data.append({
            "Year": year,
            "Fitment_Text": line
        })

    df = pd.DataFrame(data)

    # Export to CSV
    df.to_csv("porsche-tie-rod-kit-lemforder-3694601kt_output.csv", index=False)

    print("CSV exported: porsche-tie-rod-kit-lemforder-3694601kt_output.csv")
    print(df.head())

else:
    print("Active Fitment panel not found.")

driver.quit()


TimeoutException: Message: 
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x6a4103
	0x6a4144
	0x4ae71d
	0x4fa03d
	0x4fa41b
	0x5417f2
	0x51c954
	0x53ee17
	0x51c706
	0x4eda30
	0x4eed54
	0x9157b4
	0x91098a
	0x6cc392
	0x6bc4c8
	0x6c324d
	0x6ac478
	0x6ac63c
	0x6967ca
	0x76fcfcc9
	0x77b482ae
	0x77b4827e


In [8]:
#Not working 100% as of 11/21/2025
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

# === SCRAPE ONE URL ===
url = "https://www.fcpeuro.com/products/porsche-tie-rod-kit-lemforder-3694601kt#fitment"

# Selenium setup
options = webdriver.ChromeOptions()
#options.add_argument("--headless")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--headless=new")  # use new headless mode
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

print(f"Scraping: {url}")
driver.get(url)

wait = WebDriverWait(driver, 20)

# Scroll to force lazy-loading of fitment table
driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
time.sleep(2)

all_data = []

try:
    # Wait for the application rows to load
    wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, "div.fitmentGuide__application")
    ))

    # Additional safety delay
    time.sleep(1)

    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")

    app_rows = soup.select("div.fitmentGuide__application")

    if not app_rows:
        all_data.append({
            "URL": url,
            "Year": "",
            "Fitment_Application": "No application rows found",
            "Fitment_Additional": ""
        })
    else:
        for app in app_rows:
            app_text = app.get_text(" ", strip=True)

            # Extract Year (any 2000–2099)
            year_match = re.match(r"(^20\d{2})", app_text)
            year = year_match.group(1) if year_match else ""

            # Match additional info block under it
            additional = app.find_next_sibling("div", class_="fitmentGuide__additional")
            add_text = additional.get_text(" ", strip=True) if additional else ""

            all_data.append({
                "URL": url,
                "Year": year,
                "Fitment_Application": app_text,
                "Fitment_Additional": add_text
            })

except Exception as e:
    print("Error:", e)
    all_data.append({
        "URL": url,
        "Year": "",
        "Fitment_Application": f"Error: {e}",
        "Fitment_Additional": ""
    })

driver.quit()

# Export CSV
df = pd.DataFrame(all_data)
df.to_csv("porsche-tie-rod-kit-lemforder-3694601kt_output.csv", index=False)

print("CSV exported: porsche-tie-rod-kit-lemforder-3694601kt_output.csv")
print(df.head())


Scraping: https://www.fcpeuro.com/products/porsche-tie-rod-kit-lemforder-3694601kt#fitment
Error: Message: 
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x6a4103
	0x6a4144
	0x4ae71d
	0x4fa03d
	0x4fa41b
	0x5417f2
	0x51c954
	0x53ee17
	0x51c706
	0x4eda30
	0x4eed54
	0x9157b4
	0x91098a
	0x6cc392
	0x6bc4c8
	0x6c324d
	0x6ac478
	0x6ac63c
	0x6967ca
	0x76fcfcc9
	0x77b482ae
	0x77b4827e

CSV exported: porsche-tie-rod-kit-lemforder-3694601kt_output.csv
                                                 URL Year  \
0  https://www.fcpeuro.com/products/porsche-tie-r...        

                                 Fitment_Application Fitment_Additional  
0  Error: Message: \nStacktrace:\nSymbols not ava...                     


In [7]:
#Working as of 11/21/2025
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

# Product URL
url = "https://www.fcpeuro.com/products/audi-spark-plug-kit-bosch-101905631gkt2#fitment"

options = webdriver.ChromeOptions()
options.add_argument("--headless")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
driver.get(url)

wait = WebDriverWait(driver, 15)

# Scroll to ensure fitment panel loads
driver.execute_script("window.location.hash = 'fitment';")
driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
time.sleep(2)

# Wait for the active fitment panel
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.tabs-panel.is-active")))
time.sleep(5)

html = driver.page_source
soup = BeautifulSoup(html, "html.parser")

active_panel = soup.select_one("div.tabs-panel.is-active")
data = []

if active_panel:

    # Loop through each fitment application
    applications = active_panel.select("div.fitmentGuide__application")

    for app in applications:
        app_text = app.get_text(" ", strip=True)

        # Extract Year from start of text
        match = re.match(r"(^20\d{2})", app_text)
        year = match.group(1) if match else ""

        # Look for additional info
        next_tag = app.find_next_sibling()
        fit_add = ""
        if next_tag and "fitmentGuide__additional" in next_tag.get("class", []):
            fit_add = next_tag.get_text(" ", strip=True)

        data.append({
            "URL": url,
            "Year": year,
            "Fitment_Application": app_text,
            "Fitment_Additional": fit_add
        })

    # Fallback row if nothing found
    if not data:
        data.append({
            "URL": url,
            "Year": "",
            "Fitment_Application": "No application rows found",
            "Fitment_Additional": ""
        })

    df = pd.DataFrame(data)
    output_file = "scrape_output/porsche-tie-rod-kit-lemforder-3694601kt_output.csv"
    df.to_csv(output_file, index=False)
    print(f"CSV exported: {output_file}")
    print(df.head())

else:
    print("Active Fitment panel not found.")

driver.quit()


TimeoutException: Message: 
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x6a4103
	0x6a4144
	0x4ae71d
	0x4fa03d
	0x4fa41b
	0x5417f2
	0x51c954
	0x53ee17
	0x51c706
	0x4eda30
	0x4eed54
	0x9157b4
	0x91098a
	0x6cc392
	0x6bc4c8
	0x6c324d
	0x6ac478
	0x6ac63c
	0x6967ca
	0x76fcfcc9
	0x77b482ae
	0x77b4827e
